# **Session 2:** Camera Calibration 📷⚙️

## 0. Setup

In [1]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


## **Part A: Camera calibration** (right and left)

In this part we calibrate the two cameras of a stereo system, using the images in the ``left`` and ``right`` folders. We start with the ``left`` folder and then repeat the process with the images in the ``right`` folder. Both folders contain images of the calibration pattern in different positions and orientations, which are the input data.

The steps to calibrate a camera are:

1. Define and run the ``load_images()`` method to load the images.
2. Detect the pattern corners with ``cv2.findChessboardCorners()``. Refine the detections with ``cv2.cornerSubPix()``.
3. Check that the detections are correct by drawing them with ``cv2.drawChessboardCorners()``.
4. Define and run the ``get_chessboard_points(chessboard_shape, dx, dy)`` method, which returns the 3D coordinates of the pattern corners. The reference frame must be attached to the pattern itself.
5. Use ``cv2.calibrateCamera`` to obtain the calibration parameters of the left camera.

### **Task A.1:** Define and run the ``load_images()`` method to load the images.

In [2]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [3]:
# Build a list containing the paths of all images from the left camera
imgs_path = sorted(glob.glob(os.path.join("..", "data", "left", "*.jpg")))
print(f"{len(imgs_path)} images found")
imgs = load_images(imgs_path)

19 imágenes encontradas


### **Task A.2:** Detect the pattern corners with ``cv2.findChessboardCorners()``. Refine the detections with ``cv2.cornerSubPix()``.

In [4]:
# TODO Find corners with cv2.findChessboardCorners()
# Hint: you can use a list comprehension to process all images
pattern_size = (8, 6)  # inner corners (columns, rows)
corners = [cv2.findChessboardCorners(img, pattern_size) for img in imgs]

In [5]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

# Refine only the images with a successful detection (cor[0] == True)
corners_refined = [
    cv2.cornerSubPix(gray, cor[1], (8, 6), (-1, -1), criteria) if cor[0] else []
    for gray, cor in zip(imgs_gray, corners_copy)]

print(f"{sum(cor[0] for cor in corners)}/{len(corners)} successful detections")

19/19 detecciones correctas


### **Task A.3:** Check that the detections are correct by drawing them with ``cv2.drawChessboardCorners()``

In [6]:
imgs_copy = copy.deepcopy(imgs)

In [7]:
# TODO Use cv2.drawChessboardCorners() to draw the cornes
imgs_copy = [
    cv2.drawChessboardCorners(img, pattern_size, cor_ref, cor[0])
    for img, cor_ref, cor in zip(imgs_copy, corners_refined, corners)
]

In [8]:
# TODO Show images and save when needed

def show_image(img, title="Image"):
    cv2.imshow(title, img)
    cv2.waitKey(0)  # wait for a key press
    cv2.destroyAllWindows()

def write_image(img, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, img)

### **Task A.4:** Define and run the ``get_chessboard_points(chessboard_shape, dx, dy)`` method

In [9]:
# TODO Design the method. It should return a np.array with np.float32 elements
def get_chessboard_points(chessboard_shape, dx, dy):
    cols, rows = chessboard_shape  # (8, 6): 8 corners per row, 6 rows
    return np.array(
        [[i * dx, j * dy, 0] for i in range(rows) for j in range(cols)],
        dtype=np.float32
    )

In [10]:
# TODO You need the points for every image, not just one (consider a list comprehension)
chessboard_points = [get_chessboard_points(pattern_size, 30, 30)
                     for cor in corners_refined if len(cor) > 0]

### **Task A.5:** Use ``cv2.calibrateCamera()`` to obtain the calibration parameters of the left camera

In [12]:
# Filter data and get only those with adequate detections
valid_corners = [cor for cor in corners_refined if len(cor) > 0]
# Convert list to numpy array with np.float32 elements
valid_corners = np.asarray(valid_corners, dtype=np.float32)

In [13]:
# TODO
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    chessboard_points,
    valid_corners,
    imgs_gray[0].shape[::-1],
    None,
    None
)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [14]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[424.49980412   0.         155.58449296]
 [  0.         426.73945888 129.55971706]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-5.27421036e-02 -1.46215131e+00  4.83483444e-03 -5.90090044e-03
   6.02537794e+00]]
Root mean squared reprojection error:
 0.1847854065061554


### **Question A.1:** Repeat the process (image loading, corner detection and checking, etc.) for the right camera.

In [15]:
# 1. Load the right camera images
imgs_path_r = sorted(glob.glob(os.path.join("..", "data", "right", "*.jpg")))
imgs_r = load_images(imgs_path_r)

# 2. Corner detection and refinement
corners_r = [cv2.findChessboardCorners(img, pattern_size) for img in imgs_r]
corners_copy_r = copy.deepcopy(corners_r)
imgs_gray_r = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs_r]
corners_refined_r = [
    cv2.cornerSubPix(gray, cor[1], (8, 6), (-1, -1), criteria) if cor[0] else []
    for gray, cor in zip(imgs_gray_r, corners_copy_r)]

print(f"{sum(cor[0] for cor in corners_r)}/{len(corners_r)} successful detections")
print("No detection:", [os.path.basename(p) for p, cor in zip(imgs_path_r, corners_r) if not cor[0]])

# 3. Check: draw the corners and save the images
imgs_copy_r = copy.deepcopy(imgs_r)
imgs_copy_r = [
    cv2.drawChessboardCorners(img, pattern_size, cor_ref, cor[0]) if cor[0] else img
    for img, cor_ref, cor in zip(imgs_copy_r, corners_refined_r, corners_r)]
for path, img in zip(imgs_path_r, imgs_copy_r):
    write_image(img, os.path.join("..", "results", "right", os.path.basename(path)))

# 4. 3D pattern points (only for images with a successful detection)
valid_corners_r = [cor for cor in corners_refined_r if len(cor) > 0]
chessboard_points_r = [get_chessboard_points(pattern_size, 30, 30) for _ in valid_corners_r]
valid_corners_r = np.asarray(valid_corners_r, dtype=np.float32)

# 5. Calibration
rms_r, intrinsics_r, dist_coeffs_r, rvecs_r, tvecs_r = cv2.calibrateCamera(
    chessboard_points_r, valid_corners_r, imgs_gray_r[0].shape[::-1], None, None)
extrinsics_r = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs_r, tvecs_r))

print("Intrinsics:\n", intrinsics_r)
print("Distortion coefficients:\n", dist_coeffs_r)
print("Root mean squared reprojection error:\n", rms_r)

17/19 detecciones correctas
Sin detección: ['right_001.jpg', 'right_014.jpg']
Intrinsics:
 [[429.74668585   0.         146.02757684]
 [  0.         430.92875981 137.87387707]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-0.11260577 -0.32047562  0.00637188 -0.00524419  2.19809693]]
Root mean squared reprojection error:
 0.1028161909271028


### **Question A.2:** Differences between cv2.findChessboardCorners() and cv2.cornerSubPix()

**``cv2.findChessboardCorners()``: global pattern detection**

- **Input:** the full image (colour or grayscale) and the number of inner corners of the pattern (``pattern_size = (8, 6)``).
- **What it does:** binarises the image, finds the black squares as quadrilaterals and checks that they form a grid of the given size. The corners are taken from the points where the squares touch.
- **Output:** a boolean ``ret`` indicating whether the **complete** pattern was found, and the corner coordinates **ordered** row by row (needed to match them with the 3D points).
- **Accuracy:** in the order of **1 pixel**.

**``cv2.cornerSubPix()``: local sub-pixel refinement**

- **Input:** the **grayscale** image, the already detected corners, a search window size and a stopping criterion.
- **What it does:** it detects nothing; it only **improves** the corners it receives. It relies on the fact that, at the true corner $q$, the image gradient at any point $p$ in its neighbourhood is perpendicular to the vector $q - p$, i.e. $\nabla I(p)^T (q - p) = 0$. It solves this system by least squares within the window and iterates until the criterion is met (``EPS`` or ``MAX_ITER``).
- **Output:** the same corners with **sub-pixel** accuracy (tenths or hundredths of a pixel).

**Summary of differences**

| | ``findChessboardCorners`` | ``cornerSubPix`` |
|---|---|---|
| Type | Global detector | Local refiner |
| Needs prior corners | No | Yes |
| Input image | Colour or grayscale | Grayscale only (uses gradients) |
| Knows it is a chessboard | Yes | No (works for any corner) |
| Accuracy | ~1 px | Sub-pixel |

**Experimental results** (next cell):

| Camera | Mean shift | Max shift | RMS unrefined | RMS refined |
|---|---|---|---|---|
| Left | 0.14 px | 2.98 px | 0.1846 px | 0.1848 px |
| Right | 0.10 px | 1.14 px | 0.1457 px | 0.1028 px |

On average, refinement only moves the corners by a tenth of a pixel, because in OpenCV 4.x ``findChessboardCorners`` already applies a light internal refinement. Even so, for the right camera it reduces the reprojection error by 30 %. For the left camera the difference is negligible, which shows that the initial detections were already good. Since the cost is minimal and it can noticeably improve the calibration, it is worth always applying it before calibrating.

In [ ]:
# Comparison: findChessboardCorners() corners vs corners refined with cornerSubPix()
def compare_refinement(name, corners, corners_refined, imgs_gray):
    raw = [cor[1] for cor in corners if cor[0]]
    ref = [cor for cor in corners_refined if len(cor) > 0]
    dist = np.concatenate([np.linalg.norm(r - c, axis=2).ravel() for r, c in zip(ref, raw)])
    points = [get_chessboard_points(pattern_size, 30, 30) for _ in raw]
    size = imgs_gray[0].shape[::-1]
    rms_raw = cv2.calibrateCamera(points, np.asarray(raw, dtype=np.float32), size, None, None)[0]
    rms_ref = cv2.calibrateCamera(points, np.asarray(ref, dtype=np.float32), size, None, None)[0]
    print(f"{name}: mean shift {dist.mean():.3f} px, max {dist.max():.3f} px | "
          f"RMS unrefined {rms_raw:.4f} px, refined {rms_ref:.4f} px")

compare_refinement("Left", corners, corners_refined, imgs_gray)
compare_refinement("Right", corners_r, corners_refined_r, imgs_gray_r)

### **Question A.3:** Minimum number of images needed for calibration.

In [ ]:
# TODO Homework

## **Part B: Distortion correction** (fisheye)

In this part we correct the distortion caused by fisheye lenses. First we calibrate a camera with this type of lens using the images in the ``fisheye`` folder. Then we use these calibration parameters to correct the distortion of one of the calibration images.

The steps to calibrate a camera with fisheye distortion are:

1. Reuse the ``load_images()`` method to load the images in the ``fisheye`` folder.
2. Detect the corners by processing the images with ``cv2.findChessboardCorners()`` and ``cv2.cornerSubPix()``.
3. Reuse the ``get_chessboard_points()`` function to obtain the chessboard coordinates.
4. Define the arguments for the calibration function.
5. Calibrate with ``cv2.fisheye.calibrate()``

### **Task B.1:** Reuse the ``load_images()`` method to load the images in the ``fisheye`` folder

In [9]:
# TODO Build a list containing the paths of all images from the fisheye camera and load images
fisheye_imgs_path = sorted(glob.glob(os.path.join("..", "data", "fisheye", "*.jpg")))
fisheye_imgs = load_images(fisheye_imgs_path)

### **Task B.2:** Detect the corners by processing the images with ``cv2.findChessboardCorners()`` and ``cv2.cornerSubPix()``.

In [10]:
imgs_corners = []
# Parameters for cv2.cornerSubPix()
subpix_criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)

#TODO Complete the required parts of the loop
for img in fisheye_imgs:

    # TODO parse arguments to cv2.findChessboardCorners()
    ret, corners = cv2.findChessboardCorners(image=img, patternSize=(7,6))

    # TODO convert image to grayscale to use cv2.cornerSubPix()
    gray_img = cv2.cvtColor(src=img, code= cv2.COLOR_BGR2GRAY)

    # TODO append only those refined_corners with proper detections
    if ret:
        refined_corners = cv2.cornerSubPix(gray_img, corners, (3,3), (-1,-1), subpix_criteria)
        imgs_corners.append(refined_corners)

print(f"Valid detections: {len(imgs_corners)}/{len(fisheye_imgs)}")

Detecciones válidas: 10/10


### **Task B.3:** Reuse the ``get_chessboard_points()`` function to obtain the chessboard coordinates

In [11]:
# TODO Define the chessboard dimensions and the lenght of the squares (in [mm])
chessboard_dims = (7, 6)
length = 30
# TODO You need the points for every image, not just one (consider a list comprehension)
# cv2.fisheye.calibrate() expects each array with shape (n, 1, 3)
fisheye_chessboard_points = [get_chessboard_points(chessboard_dims, length, length).reshape(-1, 1, 3) for _ in imgs_corners]

NameError: name 'get_chessboard_points' is not defined

### **Task B.4:** Define the calibration arguments

In [ ]:
# Parameters for cv2.fisheye.calibrate()
calibration_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC+cv2.fisheye.CALIB_FIX_SKEW
intrinsics = np.zeros((3, 3))
distortion = np.zeros((4, 1))
rotations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]
traslations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]


### **Task B.5:** Calibration

In [ ]:
rms, intrinsics, distortion, rotations, traslations = cv2.fisheye.calibrate(fisheye_chessboard_points, imgs_corners, gray_img.shape[::-1], intrinsics, distortion, rotations, traslations, calibration_flags, subpix_criteria)

In [ ]:
# Show intrinsic matrix and distortion coefficients values
print(intrinsics)
print(distortion)
print(rms)

### **Question B.1:** Correct the distortion of the first 2 images in the ``fisheye`` folder

In [ ]:
# TODO Search in the documentation to define 'dim'
# 'dim' is the size (width, height) of the undistorted output image: we keep the original size
dim = gray_img.shape[::-1]
map1, map2 = cv2.fisheye.initUndistortRectifyMap(intrinsics, distortion, np.eye(3), intrinsics, dim, cv2.CV_16SC2)

In [ ]:
# TODO Homework: correct distortion using cv2.remap()
import matplotlib.pyplot as plt

print("Intrinsics:")
print(intrinsics)
print("Distortion coefficients:")
print(distortion)

output_dir = os.path.join("..", "results")
os.makedirs(output_dir, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (img, path) in enumerate(zip(fisheye_imgs[:2], fisheye_imgs_path[:2])):
    undistorted_img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    name = os.path.splitext(os.path.basename(path))[0]
    cv2.imwrite(os.path.join(output_dir, f"{name}_undistorted.jpg"), undistorted_img)

    axes[i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"{name} (original)")
    axes[i, 1].imshow(cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f"{name} (undistorted)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()